# Mule Pattern Learner — End-to-End Pipeline Walkthrough


## 0 · Setup & connect

In [1]:
# Make the src-layout package importable when running from the repo root.
import sys, pathlib
ROOT = pathlib.Path.cwd()
if (ROOT / "src").is_dir() and str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from mule_pattern_learner.tigergraph.client import Client
from mule_pattern_learner.tigergraph.settings import Settings

# Settings reads host / graphname / secret from .env (see tigergraph/settings.py).
client = Client(Settings())
print("connected:", client.graphname)

connected: Mule_Pattern_Learner


In [2]:
# A tiny helper to keep long gsql logs readable in the notebook.
def show(text: str, n: int = 100) -> None:
    text = text.strip()
    print(text[:n] + ("\n... [truncated]" if len(text) > n else ""))

## 1 · The GSQL installer

In [3]:
from mule_pattern_learner.tigergraph import gsql_install
from mule_pattern_learner.tigergraph.gsql_paths import query_names, gsql_path

# Show the registry-key -> installed-name map so the mismatches are explicit.
print(f"{'registry key':28s} installed name")
for key in query_names():
    try:
        print(f"{key:28s} {gsql_install.get_query_from_file(key)}")
    except gsql_install.GsqlInstallError:
        print(f"{key:28s} (not a query — e.g. loading job)")

registry key                 installed name
account_account_degree       account_aa_degree_feature
cluster_with_wcc             tg_wcc_account_with_weights
derive_max_bins              derive_max_bins
derive_reference_epoch       derive_reference_epoch
diagnose_export              diagnose_export
diagnose_sentinels           diagnose_sentinels
export_account_features      export_account_features
export_edges_by_type         export_edges_by_type
export_has_paid_edges        export_has_paid_edges
fastrp                       tg_fastRP
fetch_account_features       fetch_account_features
fetch_has_paid_features      fetch_has_paid_features
get_masking_inputs           get_masking_inputs
get_split_accounts           get_split_accounts
identity_sharing             account_identity_sharing_features
loading_job                  (not a query — e.g. loading job)
match_parties                match_parties
money_flow                   account_money_flow_features
pagerank                     tg_pag

## 2 · Schema & load — INSPECT only

In [3]:
# Read-only: list vertex & edge types and a few counts. No mutation.
vtypes = client.conn.getVertexTypes()
etypes = client.conn.getEdgeTypes()
print("vertex types:", vtypes)
print("edge types:", etypes)
print()
for vt in ("Account", "Party", "Resolved_Entity"):
    if vt in vtypes:
        cnt = client.conn.getVertexCount(vt, realtime=True)
        print(f"  {vt:18s} {cnt}")

vertex types: ['Account', 'Party', 'Device', 'IP', 'Phone', 'Email', 'Name', 'Birthdate', 'Street_Address', 'City', 'State', 'Postcode', 'Name_Minhash', 'Address_Minhash', 'Street_Minhash', 'Connected_Component', 'Resolved_Entity']
edge types: ['HAS_PAID', 'Account_Account', 'Party_Has_Account', 'Has_Device', 'Has_IP', 'Has_Phone', 'Has_Email', 'Has_Name', 'Has_Birthdate', 'Has_Street_Address', 'Has_City', 'Has_State', 'Has_Postcode', 'Has_Name_Minhash', 'Has_Address_Minhash', 'Has_Street_Minhash', 'Same_As', 'Party_In_Entity', 'Account_In_Ring']

  Account            545145
  Party              200000
  Resolved_Entity    199756


In [5]:
print("schema.gsql / loading_job.gsql exist at:")
print(" ", gsql_path("loading_job"))
print("(inspect-only — graph already built)")

schema.gsql / loading_job.gsql exist at:
  /Users/abrahamchandy/Documents/Work/companies/2026/TigerGraph/MulePatternLearner/gsql/schema/loading_job.gsql
(inspect-only — graph already built)


## 3 · Community detection (WCC)

Two stages, in order: weight the Account–Account edges, then run weighted WCC,
which materializes `Connected_Component` vertices + `Account_In_Ring` edges and
writes `com_id` / `com_size` onto Accounts. **Mutates** community attributes —
gated, but safe to re-run (idempotent overwrite).

In [6]:
RERUN = False
# Install both community queries from their .gsql files.
for key in ("weight_account_edges", "cluster_with_wcc"):
    log = gsql_install.install_query(client, key)
    print(f"installed {key} ->", gsql_install.get_query_from_file(key))
    show(log, 200)
    print("-" * 40)

ReadTimeout: HTTPSConnectionPool(host='tg-59626898-34a0-406c-a5e5-a63b57d0dda4.tg-2635877100.i.tgcloud.io', port=443): Read timed out. (read timeout=None)

In [9]:
# Run them in order (WCC depends on the weighted edges). Re-running overwrites
# com_id/com_size and re-materializes the components — idempotent, so allowed
# without the destructive gate, but flip to taste.
if RERUN:
    out1 = gsql_install.run_query(client, "weight_account_edges")
    print("weight_account_edges ->", out1)
    out2 = gsql_install.run_query(client, "cluster_with_wcc", {"min_link_weight": 2})
    print("cluster_with_wcc ->", out2)
else:
    print("Skipped run (RUN_DESTRUCTIVE=False). Communities already materialized.")
    # Inspect existing communities instead:
    sample = client.conn.runInstalledQuery  # already-installed query call site
    print("com_size distribution (sample of accounts):")
    df = client.conn.getVertexDataFrame("Account", select="com_id,com_size", limit=5)
    print(df)

Skipped run (RUN_DESTRUCTIVE=False). Communities already materialized.
com_size distribution (sample of accounts):
          v_id  com_id  com_size
0  A0000076849       0    429438
1  A0000161351       0    429438
2   XM00001231       0    429438
3  A0000053692       0    429438
4   XM00002314       0    429438


## 4 · Entity resolution

`match_parties` writes `Same_As` edges (scored party↔party matches), then
`unify_parties` collapses connected parties into `Resolved_Entity` vertices via
`Party_In_Entity`. **Mutates** entity edges/vertices — gated.

In [4]:
for key in ("match_parties", "unify_parties"):
    log = gsql_install.install_query(client, key)
    print(f"installed {key}")
    show(log, 200)
    print("-" * 40)

installed match_parties
Using graph 'Mule_Pattern_Learner'
Successfully created queries: [match_parties].
Start installing queries, about 1 minute ...
match_parties query: curl -X GET 'http://tg-59626898-34a0-406c-a5e5-a63b5
... [truncated]
----------------------------------------
installed unify_parties
Using graph 'Mule_Pattern_Learner'
Successfully created queries: [unify_parties].
Start installing queries, about 1 minute ...
unify_parties query: curl -X GET 'http://tg-59626898-34a0-406c-a5e5-a63b5
... [truncated]
----------------------------------------


In [ ]:
if RERUN:
    m = gsql_install.run_query(client, "match_parties")
    print("match_parties ->", str(m)[:300])
    u = gsql_install.run_query(client, "unify_parties")
    print("unify_parties ->", str(u)[:300])
else:
    n_ent = client.conn.getVertexCount("Resolved_Entity", realtime=True)
    print(f"Skipped (RUN_DESTRUCTIVE=False). Resolved_Entity count: {n_ent}")

## 5 · Feature queries

Seven account-level feature queries. All **overwrite attributes** on existing
Accounts (idempotent), so re-running is safe. They have no ordering dependency
on each other, but all should run before deriving the temporal spec.

In [ ]:
FEATURE_QUERIES = [
    "pagerank",
    "fastrp",
    "money_flow",
    "temporal_features",
    "identity_sharing",
    "triangle_clustering",
    "account_account_degree",
]

# Install all feature queries (resolves each to its real installed name).
logs = gsql_install.install_queries(client, FEATURE_QUERIES)
for key in FEATURE_QUERIES:
    print(f"installed {key:24s} -> {gsql_install.installed_query_name(key)}")

In [ ]:
# Run them. Each writes its features onto Account vertices. Safe to re-run.
# (Some are heavy full-graph scans — expect minutes each on the full graph.)
RUN_FEATURES = False  # flip to True to actually (re)compute features
if RUN_FEATURES:
    for key in FEATURE_QUERIES:
        out = gsql_install.run_query(client, key)
        print(f"{key:24s} done; sample output: {str(out)[:120]}")
else:
    print("Skipped feature runs (RUN_FEATURES=False). Features already on the graph.")
    df = client.conn.getVertexDataframe(
        "Account", select="pagerank,aa_degree,triangle_count,out_amount", limit=5
    )
    print(df)

## 6 · Data Preprocessing 

### Features per edge

In [11]:
from mule_pattern_learner.features.edge_spec import (
    SCALAR_FEATURE_NAMES,
    NUM_SCALAR_FEATURES,
    NUM_BIN_CHANNELS,
    flat_edge_dim,
)

print("Each HAS_PAID edge is described by two kinds of features:\n")

print(f"1. SCALAR features — {NUM_SCALAR_FEATURES} single features, one each:")
for name in SCALAR_FEATURE_NAMES:
    print(f"     • {name}")
print()

print(f"2. BIN features — {NUM_BIN_CHANNELS} channels (amount + count), one value PER time-bin.")
print("   If the graph uses, say, 13 time-bins, that's 13 amount-values + 13 count-values.\n")

for bins in (1, 13, 100):
    scalars = NUM_SCALAR_FEATURES
    bin_part = bins * NUM_BIN_CHANNELS
    total = flat_edge_dim(bins)
    print(f"   with {bins:>3} bins:  {scalars} scalars + ({bins} bins × {NUM_BIN_CHANNELS} channels = {bin_part})  =  {total} features per edge")

print(f"\nSo flat_edge_dim(max_bins) = {NUM_SCALAR_FEATURES} + max_bins × {NUM_BIN_CHANNELS}")
print("'flat' = the per-bin values are laid out in ONE long list, not a 2-D grid.")

Each HAS_PAID edge is described by two kinds of features:

1. SCALAR features — 8 single features, one each:
     • amount_transferred
     • transaction_count
     • active_span_days
     • active_bin_count
     • recency_days
     • duration_days
     • active_bin_fraction
     • amount_per_transaction

2. BIN features — 2 channels (amount + count), one value PER time-bin.
   If the graph uses, say, 13 time-bins, that's 13 amount-values + 13 count-values.

   with   1 bins:  8 scalars + (1 bins × 2 channels = 2)  =  10 features per edge
   with  13 bins:  8 scalars + (13 bins × 2 channels = 26)  =  34 features per edge
   with 100 bins:  8 scalars + (100 bins × 2 channels = 200)  =  208 features per edge

So flat_edge_dim(max_bins) = 8 + max_bins × 2
'flat' = the per-bin values are laid out in ONE long list, not a 2-D grid.


In [12]:
from mule_pattern_learner.schema.specs import (
    ACCOUNT_NUMERIC_FEATURES,
    TARGET_LABEL_ATTR,
    EVAL_LABEL_ATTR,
    LEAKY_EXCLUDED_ATTRS,
    VERTEX_FEATURES,
)

# 1. The account feature list — THIS defines the model's input width.
print(f"ACCOUNT_NUMERIC_FEATURES: {len(ACCOUNT_NUMERIC_FEATURES)} features\n")
for i, name in enumerate(ACCOUNT_NUMERIC_FEATURES):
    print(f"  col {i:>2}: {name}")

# 2. The labels — what's a target, what's eval-only, what's forbidden as a feature.
print(f"\ntraining target (the model learns against this): {TARGET_LABEL_ATTR!r}")
print(f"eval-only ground truth (NEVER a feature)        : {EVAL_LABEL_ATTR!r}")
print(f"\nleakage-excluded attrs (banned from features): {len(LEAKY_EXCLUDED_ATTRS)}")
for name in LEAKY_EXCLUDED_ATTRS:
    print(f"  ✗ {name}")

# 3. Which node types even HAVE features (Account does; most others don't).
print("\nfeatures per node type:")
for vtype, feats in VERTEX_FEATURES.items():
    n = len(feats)
    detail = f"{n} features" if n else "(featureless — gets a learned embedding instead)"
    print(f"  {vtype:<16}: {detail}")

ACCOUNT_NUMERIC_FEATURES: 31 features

  col  0: pagerank
  col  1: com_size
  col  2: aa_degree
  col  3: triangle_count
  col  4: clustering_coef
  col  5: in_degree
  col  6: out_degree
  col  7: in_amount
  col  8: out_amount
  col  9: in_txn_count
  col 10: out_txn_count
  col 11: fan_in_ratio
  col 12: fan_out_ratio
  col 13: pass_through_ratio
  col 14: net_flow
  col 15: activity_span_days
  col 16: days_since_last_txn
  col 17: account_age_days
  col 18: mean_inter_txn_days
  col 19: txn_per_active_day
  col 20: burst_ratio
  col 21: active_bin_count
  col 22: activity_concentration
  col 23: peak_bin_fraction
  col 24: early_late_ratio
  col 25: in_out_lag_days
  col 26: device_share_cnt
  col 27: ip_share_cnt
  col 28: phone_share_cnt
  col 29: email_share_cnt
  col 30: is_external

training target (the model learns against this): 'pu_label'
eval-only ground truth (NEVER a feature)        : 'is_fraud'

leakage-excluded attrs (banned from features): 11
  ✗ is_fraud
  ✗ pu_lab

In [13]:
from mule_pattern_learner.features.nodes import (
    Transform,
    _ACCOUNT_FEATURE_TRANSFORMS,
    _apply_transform,
    log1p_compress,
    symlog_compress,
)

# 1. Show each transform on illustrative raw values.
print("LOG1P  (heavy-tailed non-negative counts -> compressed):")
for v in (0.0, 1.0, 100.0, 100_000.0):
    print(f"    {v:>12,.1f}  ->  {log1p_compress(v):>8.4f}")

print("\nSYMLOG (signed heavy-tailed, e.g. net_flow -> sign-preserving compress):")
for v in (-100_000.0, -10.0, 0.0, 10.0, 100_000.0):
    print(f"    {v:>12,.1f}  ->  {symlog_compress(v):>8.4f}")

print("\nCLAMP_SENTINEL (-1.0 means 'missing' -> becomes 0.0; else unchanged):")
for v in (-1.0, 0.0, 42.0):
    print(f"    {v:>12,.1f}  ->  {_apply_transform(v, Transform.CLAMP_SENTINEL):>8.4f}")

print("\nBOOLEAN (anything nonzero -> 1.0):")
for v in (0.0, 1.0, 5.0):
    print(f"    {v:>12,.1f}  ->  {_apply_transform(v, Transform.BOOLEAN):>8.4f}")

# 2. Which transform is applied to each of the 31 features?
print("\nper-feature transform policy:")
for name, t in _ACCOUNT_FEATURE_TRANSFORMS.items():
    print(f"    {name:<24} {t.name}")

LOG1P  (heavy-tailed non-negative counts -> compressed):
             0.0  ->    0.0000
             1.0  ->    0.6931
           100.0  ->    4.6151
       100,000.0  ->   11.5129

SYMLOG (signed heavy-tailed, e.g. net_flow -> sign-preserving compress):
      -100,000.0  ->  -11.5129
           -10.0  ->   -2.3979
             0.0  ->    0.0000
            10.0  ->    2.3979
       100,000.0  ->   11.5129

CLAMP_SENTINEL (-1.0 means 'missing' -> becomes 0.0; else unchanged):
            -1.0  ->    0.0000
             0.0  ->    0.0000
            42.0  ->   42.0000

BOOLEAN (anything nonzero -> 1.0):
             0.0  ->    0.0000
             1.0  ->    1.0000
             5.0  ->    1.0000

per-feature transform policy:
    pagerank                 IDENTITY
    com_size                 LOG1P
    aa_degree                LOG1P
    triangle_count           LOG1P
    clustering_coef          IDENTITY
    in_degree                LOG1P
    out_degree               LOG1P
    in_amount  

In [14]:
from mule_pattern_learner.features.nodes import (
    build_account_features,
    account_feature_names,
    NUM_ACCOUNT_FEATURES,
)
from mule_pattern_learner.schema.specs import ACCOUNT_NUMERIC_FEATURES

# Two fake accounts, shaped exactly like TigerGraph vertex rows
fake_vertices = [
    {
        "v_id": "ACC_AAA",
        "attributes": {
            "in_amount": 100000.0,  
            "net_flow": -50000.0,  
            "days_since_last_txn": -1.0,
            "is_external": 1.0, 
            "pagerank": 0.0003, 
        },
    },
    {
        "v_id": "ACC_BBB",
        "attributes": {
            "in_amount": 0.0,
            "net_flow": 250.0, 
            "days_since_last_txn": 30.0,
            "is_external": 0.0,
            "pagerank": 0.01,
        },
    },
]

result = build_account_features(fake_vertices)

print(f"input: {len(fake_vertices)} fake vertices")
print(f"output type: {type(result).__name__}")
print(f"output tensor shape: {tuple(result.feats.shape)}  (expect (2, {NUM_ACCOUNT_FEATURES}))")
print(f"output dtype: {result.feats.dtype}")
print(f"node_ids: {result.node_ids}")
print()

# Show raw -> transformed for the features we gave distinctive values.
watch = ["in_amount", "net_flow", "days_since_last_txn", "is_external", "pagerank"]
col = {name: i for i, name in enumerate(ACCOUNT_NUMERIC_FEATURES)}
print(f"{'feature':<22}{'raw (ACC_AAA)':>16}{'transformed':>16}")
for name in watch:
    raw = fake_vertices[0]["attributes"][name]
    transformed = float(result.feats[0, col[name]])
    print(f"{name:<22}{raw:>16,.4f}{transformed:>16.4f}")

input: 2 fake vertices
output type: NodeFeatures
output tensor shape: (2, 31)  (expect (2, 31))
output dtype: torch.float32
node_ids: ('ACC_AAA', 'ACC_BBB')

feature                  raw (ACC_AAA)     transformed
in_amount                 100,000.0000         11.5129
net_flow                  -50,000.0000        -10.8198
days_since_last_txn            -1.0000          0.0000
is_external                     1.0000          1.0000
pagerank                        0.0003          0.0003


In [16]:
import torch
from mule_pattern_learner.features.nodes import (
    normalizer_from_features,
    FeatureNormalizer,
    NUM_ACCOUNT_FEATURES,
)

# A tiny fake "train" feature matrix: 5 accounts x 31 features, already transformed
torch.manual_seed(0)
fake_train = torch.randn(5, NUM_ACCOUNT_FEATURES)
fake_train[:, 0] = torch.tensor([11.0, 11.5, 12.0, 11.2, 11.8])
fake_train[:, 1] = torch.tensor([0.1, 0.2, 0.15, 0.3, 0.25])

# FIT: compute per-feature mean and std from train.
normalizer = normalizer_from_features(fake_train)
print(f"fitted normalizer on {fake_train.shape[0]} accounts")
print(f"  mean shape: {tuple(normalizer.mean.shape)}  (expect ({NUM_ACCOUNT_FEATURES},))")
print(f"  std shape : {tuple(normalizer.std.shape)}")
print()
print(f"col 0 (log-amounts ~11-12):  mean={normalizer.mean[0]:.4f}  std={normalizer.std[0]:.4f}")
print(f"col 1 (ratio ~0-0.3):        mean={normalizer.mean[1]:.4f}  std={normalizer.std[1]:.4f}")
print()

# APPLY: standardize. Each column becomes ~zero-mean, ~unit-spread.
standardized = normalizer.apply(fake_train)
print("BEFORE standardization (col 0 raw values):")
print(f"  {fake_train[:, 0].tolist()}")
print("AFTER standardization (col 0 -> centered on 0, scaled by std):")
print(f"  {[round(v, 4) for v in standardized[:, 0].tolist()]}")
print()
print(f"col 0 after:  mean={standardized[:, 0].mean():.4f} (≈0)  std={standardized[:, 0].std(unbiased=False):.4f} (≈1)")
print(f"col 1 after:  mean={standardized[:, 1].mean():.4f} (≈0)  std={standardized[:, 1].std(unbiased=False):.4f} (≈1)")

fitted normalizer on 5 accounts
  mean shape: (31,)  (expect (31,))
  std shape : (31,)

col 0 (log-amounts ~11-12):  mean=11.5000  std=0.3688
col 1 (ratio ~0-0.3):        mean=0.2000  std=0.0707

BEFORE standardization (col 0 raw values):
  [11.0, 11.5, 12.0, 11.199999809265137, 11.800000190734863]
AFTER standardization (col 0 -> centered on 0, scaled by std):
  [-1.3558, 0.0, 1.3558, -0.8135, 0.8135]

col 0 after:  mean=0.0000 (≈0)  std=1.0000 (≈1)
col 1 after:  mean=0.0000 (≈0)  std=1.0000 (≈1)


In [17]:
import torch
from mule_pattern_learner.features.nodes import NodeFeatures, NodeFeatureError, NUM_ACCOUNT_FEATURES

# A VALID NodeFeatures: 2 nodes, 31 features each, names length 31.
names = tuple(f"feat_{i}" for i in range(NUM_ACCOUNT_FEATURES))
good = NodeFeatures(
    node_ids=("ACC_A", "ACC_B"),
    feats=torch.zeros(2, NUM_ACCOUNT_FEATURES),
    feature_names=names,
)
print("valid NodeFeatures built:")
print(f"  num_nodes     : {good.num_nodes}")
print(f"  feats shape   : {tuple(good.feats.shape)}")
print(f"  feature_names : {len(good.feature_names)} names")
print()

# INVALID: tensor says 3 rows but only 2 ids -> guard must raise.
print("MISMATCHED shape (3 tensor rows, 2 ids):")
try:
    NodeFeatures(
        node_ids=("ACC_A", "ACC_B"),            # 2 ids
        feats=torch.zeros(3, NUM_ACCOUNT_FEATURES),  # but 3 rows
        feature_names=names,
    )
except NodeFeatureError as e:
    print(f"  ✓ raised NodeFeatureError: {e}")

# INVALID: wrong feature-count -> guard must raise.
print("\nwrong feature width (tensor has 10 cols, names has 31):")
try:
    NodeFeatures(
        node_ids=("ACC_A", "ACC_B"),
        feats=torch.zeros(2, 10),                # 10 cols
        feature_names=names,                      # but 31 names
    )
except NodeFeatureError as e:
    print(f"  ✓ raised NodeFeatureError: {e}")

valid NodeFeatures built:
  num_nodes     : 2
  feats shape   : (2, 31)
  feature_names : 31 names

MISMATCHED shape (3 tensor rows, 2 ids):
  ✓ raised NodeFeatureError: feats shape (3, 31) != (2, 31)

wrong feature width (tensor has 10 cols, names has 31):
  ✓ raised NodeFeatureError: feats shape (2, 10) != (2, 31)


In [18]:
from mule_pattern_learner.features.temporal import _as_epoch_seconds

print("epoch-number input (passed through as float):")
for v in (0, 1753450140, 1753450140.0):
    print(f"    {v!r:>20}  ->  {_as_epoch_seconds(v, 'test'):>16.1f}")

print("\ndatetime-STRING input (parsed as UTC -> epoch seconds):")
for v in ("2025-07-25 14:09:00", "1970-01-01 00:00:00", "2024-01-01 00:00:00"):
    print(f"    {v!r:>22}  ->  {_as_epoch_seconds(v, 'test'):>16.1f}")

# Show the round-trip so the number is meaningful, not magic.
from datetime import datetime, timezone
epoch = _as_epoch_seconds("2025-07-25 14:09:00", "test")
back = datetime.fromtimestamp(epoch, tz=timezone.utc)
print(f"\nround-trip check: '2025-07-25 14:09:00' -> {epoch:.0f} -> {back} (UTC)")

# And show it rejects garbage rather than silently producing a wrong number.
try:
    _as_epoch_seconds("not-a-date", "test")
except Exception as e:
    print(f"\nrejects bad input: {type(e).__name__}: {str(e)[:80]}")

epoch-number input (passed through as float):
                       0  ->               0.0
              1753450140  ->      1753450140.0
            1753450140.0  ->      1753450140.0

datetime-STRING input (parsed as UTC -> epoch seconds):
     '2025-07-25 14:09:00'  ->      1753452540.0
     '1970-01-01 00:00:00'  ->               0.0
     '2024-01-01 00:00:00'  ->      1704067200.0

round-trip check: '2025-07-25 14:09:00' -> 1753452540 -> 2025-07-25 14:09:00+00:00 (UTC)

rejects bad input: EdgeFeatureError: test: could not parse datetime string 'not-a-date' with format '%Y-%m-%d %H:%M:%


##### test padding

In [19]:
from mule_pattern_learner.features.temporal import _pad_bins

MAX_BINS = 5  # small, so padding/truncation is easy to see

print(f"target width (max_bins) = {MAX_BINS}\n")

# SHORT list -> padded with zeros to max_bins.
short = [10.0, 20.0]
padded, raw_len = _pad_bins(short, "amount_bins", MAX_BINS)
print(f"short input {short} (len {len(short)}):")
print(f"  -> padded: {padded}   (raw_len reported: {raw_len})")
print(f"     length is now exactly {len(padded)} = max_bins")

# EXACT-length list -> unchanged.
exact = [1.0, 2.0, 3.0, 4.0, 5.0]
padded, raw_len = _pad_bins(exact, "amount_bins", MAX_BINS)
print(f"\nexact input {exact} (len {len(exact)}):")
print(f"  -> {padded}   (raw_len: {raw_len})")

# TOO-LONG list -> truncated to max_bins, but raw_len reports the TRUE length.
too_long = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
padded, raw_len = _pad_bins(too_long, "amount_bins", MAX_BINS)
print(f"\ntoo-long input {too_long} (len {len(too_long)}):")
print(f"  -> truncated: {padded}   (raw_len reported: {raw_len})")
print(f"     kept {len(padded)} bins, but raw_len={raw_len} flags that {raw_len - MAX_BINS} were DROPPED")

# Non-list input -> raises (data hygiene).
print("\nnon-list input (a string instead of a list):")
try:
    _pad_bins("not-a-list", "amount_bins", MAX_BINS)
except Exception as e:
    print(f"  ✓ raised {type(e).__name__}: {str(e)[:70]}")

target width (max_bins) = 5

short input [10.0, 20.0] (len 2):
  -> padded: [10.0, 20.0, 0.0, 0.0, 0.0]   (raw_len reported: 2)
     length is now exactly 5 = max_bins

exact input [1.0, 2.0, 3.0, 4.0, 5.0] (len 5):
  -> [1.0, 2.0, 3.0, 4.0, 5.0]   (raw_len: 5)

too-long input [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0] (len 8):
  -> truncated: [1.0, 2.0, 3.0, 4.0, 5.0]   (raw_len reported: 8)
     kept 5 bins, but raw_len=8 flags that 3 were DROPPED

non-list input (a string instead of a list):
  ✓ raised EdgeFeatureError: amount_bins: expected list, got str: 'not-a-list'


In [20]:
from mule_pattern_learner.features.temporal import build_edge_features
from mule_pattern_learner.features.edge_spec import (
    SCALAR_FEATURE_NAMES,
    NUM_SCALAR_FEATURES,
    NUM_BIN_CHANNELS,
)

MAX_BINS = 5

REFERENCE_EPOCH_S = 1753450140.0

# Two fake edges, shaped like TigerGraph payment edges. Account A->B transacted
# a lot; C->D a little. Note last_txn_date as a STRING (cell 7 parses it).
fake_edges = [
    {
        "from_id": "ACC_A",
        "to_id": "ACC_B",
        "attributes": {
            "total_amount": 100000.0,
            "total_num_txns": 40.0,
            "span_days": 30.0,
            "num_bins": 5,
            "first_txn_date": "2025-06-25 00:00:00",
            "last_txn_date": "2025-07-24 00:00:00",  
            "amount_bins": [50000.0, 30000.0, 20000.0], 
            "count_bins": [20.0, 12.0, 8.0],
        },
    },
    {
        "from_id": "ACC_C",
        "to_id": "ACC_D",
        "attributes": {
            "total_amount": 500.0,
            "total_num_txns": 2.0,
            "span_days": 1.0,
            "num_bins": 5,
            "first_txn_date": "2025-01-01 00:00:00",
            "last_txn_date": "2025-01-01 00:00:00", 
            "amount_bins": [500.0],
            "count_bins": [2.0],
        },
    },
]

ef = build_edge_features(fake_edges, REFERENCE_EPOCH_S, MAX_BINS)

print(f"input: {len(fake_edges)} edges, max_bins={MAX_BINS}\n")
print(f"src_ids       : {ef.src_ids}")
print(f"dst_ids       : {ef.dst_ids}")
print(f"scalar_feats  : {tuple(ef.scalar_feats.shape)}  (expect ({len(fake_edges)}, {NUM_SCALAR_FEATURES}))")
print(f"bin_seq       : {tuple(ef.bin_seq.shape)}  (expect ({len(fake_edges)}, {MAX_BINS}, {NUM_BIN_CHANNELS}))")
print()

# The 8 scalar features for edge A->B, named.
print("scalar features for ACC_A -> ACC_B:")
for name, val in zip(SCALAR_FEATURE_NAMES, ef.scalar_feats[0].tolist()):
    print(f"    {name:<22} {val:>12.4f}")
print()

# The bin sequence for A->B: [max_bins, 2] = per-bin (log-amount, count).
print("bin_seq for ACC_A -> ACC_B  (each row = one time-bin: [log1p(amount), count]):")
for i, (amt, cnt) in enumerate(ef.bin_seq[0].tolist()):
    note = "  <- padding (no activity)" if i >= 3 else ""
    print(f"    bin {i}: amount={amt:>8.4f}  count={cnt:>5.1f}{note}")

input: 2 edges, max_bins=5

src_ids       : ('ACC_A', 'ACC_C')
dst_ids       : ('ACC_B', 'ACC_D')
scalar_feats  : (2, 8)  (expect (2, 8))
bin_seq       : (2, 5, 2)  (expect (2, 5, 2))

scalar features for ACC_A -> ACC_B:
    amount_transferred          11.5129
    transaction_count            3.7136
    active_span_days            30.0000
    active_bin_count             3.0000
    recency_days                 1.5618
    duration_days               29.0000
    active_bin_fraction          0.6000
    amount_per_transaction    2500.0000

bin_seq for ACC_A -> ACC_B  (each row = one time-bin: [log1p(amount), count]):
    bin 0: amount= 10.8198  count= 20.0
    bin 1: amount= 10.3090  count= 12.0
    bin 2: amount=  9.9035  count=  8.0
    bin 3: amount=  0.0000  count=  0.0  <- padding (no activity)
    bin 4: amount=  0.0000  count=  0.0  <- padding (no activity)


In [21]:
from mule_pattern_learner.features.temporal import flat_edge_features, flatten_bin_seq
from mule_pattern_learner.features.edge_spec import flat_edge_dim, NUM_SCALAR_FEATURES

MAX_BINS = ef.max_bins

flat_bins = flatten_bin_seq(ef.bin_seq)
print(f"bin_seq        {tuple(ef.bin_seq.shape)}  (3D: edges x bins x channels)")
print(f"flatten_bin_seq -> {tuple(flat_bins.shape)}  (2D: edges x bins*channels = {MAX_BINS}*2)")
print()
print("ACC_A->B bins flattened (interleaved [amt0,cnt0, amt1,cnt1, ...]):")
print(f"  {[round(v, 3) for v in flat_bins[0].tolist()]}")
print()

# scalars + flat bins -> [E, k].
flat = flat_edge_features(ef)
expected_k = flat_edge_dim(MAX_BINS)
print(f"scalar_feats   {tuple(ef.scalar_feats.shape)}  ({NUM_SCALAR_FEATURES} scalars)")
print(f"flat bins      {tuple(flat_bins.shape)}  ({MAX_BINS}*2 = {MAX_BINS*2} bin values)")
print(f"flat_edge_features -> {tuple(flat.shape)}")
print()
print(f"width k = {flat.shape[1]}")
print(f"flat_edge_dim({MAX_BINS}) = {expected_k}   (= {NUM_SCALAR_FEATURES} scalars + {MAX_BINS}*2 bins)")
print(f"MATCH: {flat.shape[1] == expected_k}")
print()

# full flat vector for ACC_A->B: first 8 = scalars, rest = bins.
print("full flat edge_attr for ACC_A->B:")
print(f"  scalars [0:8] : {[round(v, 3) for v in flat[0, :NUM_SCALAR_FEATURES].tolist()]}")
print(f"  bins   [8:{expected_k}]: {[round(v, 3) for v in flat[0, NUM_SCALAR_FEATURES:].tolist()]}")

bin_seq        (2, 5, 2)  (3D: edges x bins x channels)
flatten_bin_seq -> (2, 10)  (2D: edges x bins*channels = 5*2)

ACC_A->B bins flattened (interleaved [amt0,cnt0, amt1,cnt1, ...]):
  [10.82, 20.0, 10.309, 12.0, 9.904, 8.0, 0.0, 0.0, 0.0, 0.0]

scalar_feats   (2, 8)  (8 scalars)
flat bins      (2, 10)  (5*2 = 10 bin values)
flat_edge_features -> (2, 18)

width k = 18
flat_edge_dim(5) = 18   (= 8 scalars + 5*2 bins)
MATCH: True

full flat edge_attr for ACC_A->B:
  scalars [0:8] : [11.513, 3.714, 30.0, 3.0, 1.562, 29.0, 0.6, 2500.0]
  bins   [8:18]: [10.82, 20.0, 10.309, 12.0, 9.904, 8.0, 0.0, 0.0, 0.0, 0.0]


In [22]:
from mule_pattern_learner.pyg.fetch import fetch_account_vertices

some = client.conn.getVertices("Account", limit=3)
sample_ids = [v["v_id"] for v in some]
print(f"3 real account ids from the graph: {sample_ids}\n")

vertices = fetch_account_vertices(client, sample_ids)
print(f"fetch_account_vertices returned {len(vertices)} vertices\n")

# Look at ONE raw vertex exactly as TigerGraph returns it.
import json
print("one vertex feature (the shape build_account_features expects):")
print(json.dumps(vertices[0], indent=2, default=str)[:1800])

# Confirm the empty-list short-circuit (no server call) works too.
print(f"\nempty-id short-circuit: fetch_account_vertices(client, []) -> {fetch_account_vertices(client, [])}")

3 real account ids from the graph: ['A0000076849', 'A0000161351', 'XM00001231']

fetch_account_vertices returned 3 vertices

one vertex feature (the shape build_account_features expects):
{
  "v_id": "A0000076849",
  "v_type": "Account",
  "attributes": {
    "id": "A0000076849",
    "pagerank": 8.479247,
    "com_size": 429438,
    "aa_degree": 325,
    "triangle_count": 328,
    "clustering_coef": 0.00622982,
    "in_degree": 275,
    "out_degree": 50,
    "in_amount": 19968.37,
    "out_amount": 19431.9,
    "in_txn_count": 767,
    "out_txn_count": 175,
    "fan_in_ratio": 5,
    "fan_out_ratio": 0,
    "pass_through_ratio": 0.9731337,
    "net_flow": 536.4668,
    "activity_span_days": 178.1187,
    "days_since_last_txn": 27.42917,
    "account_age_days": 205.5479,
    "mean_inter_txn_days": 1.023671,
    "txn_per_active_day": 0.9824907,
    "burst_ratio": 1.857143,
    "active_bin_count": 13,
    "activity_concentration": 0,
    "peak_bin_fraction": 0.1350033,
    "early_late_rat

In [23]:
from mule_pattern_learner.indexing.node_id_mapper import NodeIDMapper, NodeIDMapperError

mapper = NodeIDMapper()

# REGISTER: hand it global string ids, get back contiguous integers (from 0).
account_ids = ["A0000000001", "A0000000002", "A0000000003"]
ints = mapper.register("Account", account_ids)
print("register 3 Account ids:")
for sid, i in zip(account_ids, ints):
    print(f"    {sid}  ->  int {i}")
print(f"  returned integers: {ints}  (contiguous from 0)\n")

# IDEMPOTENT: re-registering existing ids returns the SAME integers, new ones extend.
again = mapper.register("Account", ["A0000000002", "A0000000099"])
print("re-register [existing A...002, new A...099]:")
print(f"    A0000000002 -> {again[0]}  (same as before: {again[0] == 1})")
print(f"    A0000000099 -> {again[1]}  (new, extends to next int)\n")

# REVERSE: integer -> string (what the FEATURE STORE does to fetch features).
print("reverse lookups (feature store uses this direction):")
print(f"    int 0 -> {mapper.to_string('Account', 0)!r}")
print(f"    ints [0,2] -> {mapper.to_strings('Account', [0, 2])}")
print(f"    string 'A0000000003' -> int {mapper.to_int('Account', 'A0000000003')}\n")

# PER-TYPE: integers are assigned independently PER node type (both start at 0).
party_ints = mapper.register("Party", ["P000000001", "P000000002"])
print("register 2 Party ids (separate integer space from Account):")
print(f"    Party integers: {party_ints}  (also start at 0 — per-type indexing)")
print(f"    Account int 0 = {mapper.to_string('Account', 0)!r}")
print(f"    Party   int 0 = {mapper.to_string('Party', 0)!r}  (different node, same int 0)\n")

# RESET: the sampler calls this each batch so the table holds ONLY this batch.
print(f"before reset: {mapper.num_nodes('Account')} Account, {mapper.num_nodes('Party')} Party")
mapper.reset()
print(f"after reset:  {mapper.num_nodes('Account')} Account, {mapper.num_nodes('Party')} Party (empty)\n")

# Out-of-range / unknown lookups raise (no silent wrong answers).
try:
    mapper.to_string("Account", 5)
except NodeIDMapperError as e:
    print(f"out-of-range int raises: {type(e).__name__}: {str(e)[:60]}")

register 3 Account ids:
    A0000000001  ->  int 0
    A0000000002  ->  int 1
    A0000000003  ->  int 2
  returned integers: [0, 1, 2]  (contiguous from 0)

re-register [existing A...002, new A...099]:
    A0000000002 -> 1  (same as before: True)
    A0000000099 -> 3  (new, extends to next int)

reverse lookups (feature store uses this direction):
    int 0 -> 'A0000000001'
    ints [0,2] -> ['A0000000001', 'A0000000003']
    string 'A0000000003' -> int 2

register 2 Party ids (separate integer space from Account):
    Party integers: [0, 1]  (also start at 0 — per-type indexing)
    Account int 0 = 'A0000000001'
    Party   int 0 = 'P000000001'  (different node, same int 0)

before reset: 4 Account, 2 Party
after reset:  0 Account, 0 Party (empty)

out-of-range int raises: NodeIDMapperError: "no string id for 'Account' int id 5"


In [24]:
from mule_pattern_learner.indexing.reindex import (
    parse_raw_result,
    reindex_neighborhood,
    edge_type_schema,
)

fake_gsql_result = [
    {"account_ids": ["A_seed1", "A_neighbor1", "A_seed2", "A_neighbor2"]},
    {"party_ids": ["P_1", "P_2"]},
    {"edges": [
        {"from_id": "A_seed1", "to_id": "A_neighbor1", "e_type": "HAS_PAID"},
        {"from_id": "A_seed1", "to_id": "P_1",         "e_type": "Account_Party"},
        {"from_id": "A_seed2", "to_id": "A_neighbor2", "e_type": "HAS_PAID"},
        {"from_id": "P_2",     "to_id": "A_seed2",     "e_type": "Party_Account"},
    ]},
]

# parse the raw blocks into a RawNeighborhood ──
raw = parse_raw_result(fake_gsql_result)
print("STEP 1 — parse_raw_result (GSQL blocks -> global ids + edges):")
print(f"  node_ids by type:")
for ntype, ids in raw.node_ids.items():
    print(f"    {ntype:<16}: {ids}")
print(f"  edges (global-id triples): {len(raw.edges)}")
for e in raw.edges:
    print(f"    {e}")
print()

# reindex to local integer indices, with seeds ordered first ──
seed_ids = ["A_seed1", "A_seed2"]   # these must end up at Account positions 0, 1
local = reindex_neighborhood(raw, seed_ids=seed_ids)

print("STEP 2 — reindex_neighborhood (global ids -> local indices, seeds first):")
print(f"  seed_ids passed: {seed_ids}")
print(f"  Account node order AFTER reindex: {local.node['Account']}")
print(f"    -> seeds are at positions 0,1: {local.node['Account'][:2] == seed_ids}")
print(f"  Party node order: {local.node['Party']}")
print()

print("  edges as LOCAL integer indices (row -> col), per edge type:")
for etype in local.row:
    rows = local.row[etype]
    cols = local.col[etype]
    print(f"    {etype}:")
    for r, c in zip(rows, cols):
        src_name = local.node[etype[0]][r]
        dst_name = local.node[etype[2]][c]
        print(f"        row={r} col={c}   ({src_name} -> {dst_name})")

STEP 1 — parse_raw_result (GSQL blocks -> global ids + edges):
  node_ids by type:
    Account         : ['A_seed1', 'A_neighbor1', 'A_seed2', 'A_neighbor2']
    Party           : ['P_1', 'P_2']
  edges (global-id triples): 4
    ('A_seed1', 'A_neighbor1', 'HAS_PAID')
    ('A_seed1', 'P_1', 'Account_Party')
    ('A_seed2', 'A_neighbor2', 'HAS_PAID')
    ('P_2', 'A_seed2', 'Party_Account')

STEP 2 — reindex_neighborhood (global ids -> local indices, seeds first):
  seed_ids passed: ['A_seed1', 'A_seed2']
  Account node order AFTER reindex: ['A_seed1', 'A_seed2', 'A_neighbor1', 'A_neighbor2']
    -> seeds are at positions 0,1: True
  Party node order: ['P_1', 'P_2']

  edges as LOCAL integer indices (row -> col), per edge type:
    ('Account', 'HAS_PAID', 'Account'):
        row=0 col=2   (A_seed1 -> A_neighbor1)
        row=1 col=3   (A_seed2 -> A_neighbor2)
    ('Account', 'Account_Party', 'Party'):
        row=0 col=0   (A_seed1 -> P_1)
    ('Party', 'Party_Account', 'Account'):
     

#### Seed from source

In [25]:
from mule_pattern_learner.training.seeds_source import fetch_split_seeds

# Pull the train split's seeds straight from the graph.
train_seeds = fetch_split_seeds(client, "train")

print(f"fetch_split_seeds(client, 'train') returned a {type(train_seeds).__name__}:")
print(f"  total account_ids : {len(train_seeds.account_ids)}")
print(f"  revealed positives (pu_label==1): {train_seeds.num_positives}")
print(f"  unlabeled          (pu_label==0): {train_seeds.num_unlabeled}")
print()

# Look at a few real ids and their pu_labels.
print("first 5 account ids and their pu_label:")
for aid in train_seeds.account_ids[:5]:
    print(f"    {aid}  ->  pu_label = {train_seeds.pu_label_of[aid]}")
print()

# Find and show an actual revealed positive (pu_label==1), since they're rare.
positives = [a for a, y in train_seeds.pu_label_of.items() if y == 1]
print(f"a revealed-positive id (pu_label==1): {positives[0] if positives else 'NONE FOUND'}")
print(f"  (these {len(positives)} are the confirmed mules the model trains on)")

fetch_split_seeds(client, 'train') returned a SplitSeeds:
  total account_ids : 380808
  revealed positives (pu_label==1): 32
  unlabeled          (pu_label==0): 380776

first 5 account ids and their pu_label:
    A0000066179  ->  pu_label = 0
    XM00000053  ->  pu_label = 0
    A0000083304  ->  pu_label = 0
    XM00001071  ->  pu_label = 0
    XM00000608  ->  pu_label = 0

a revealed-positive id (pu_label==1): A0000272438
  (these 32 are the confirmed mules the model trains on)


##### Feature Store

In [26]:
import torch
from torch_geometric.data import TensorAttr

from mule_pattern_learner.indexing.node_id_mapper import NodeIDMapper
from mule_pattern_learner.pyg.feature_store import TigerGraphFeatureStore
from mule_pattern_learner.features.nodes import normalizer_from_features, build_account_features
from mule_pattern_learner.pyg.fetch import fetch_account_vertices
from mule_pattern_learner.training.seeds_source import fetch_split_seeds

# Real ids from the graph (re-fetch here so the cell is self-contained).
train_seeds = fetch_split_seeds(client, "train")
real_ids = list(train_seeds.account_ids[:4])
print(f"4 real account ids: {real_ids}\n")

# 1. FRESH mapper, register the ids — exactly what the sampler does per batch.
#    (Fresh instance avoids stale state — the bug from the last attempt.)
mapper = NodeIDMapper()
ints = mapper.register("Account", real_ids)
print(f"mapper.register -> integers PyG would use: {ints}")
print(f"  int 0 <-> {mapper.to_string('Account', 0)}\n")

# 2. Fit a normalizer on these accounts (real fetch + build).
verts = fetch_account_vertices(client, real_ids)
normalizer = normalizer_from_features(build_account_features(verts).feats)

# 3. Feature store sharing the SAME mapper.
store = TigerGraphFeatureStore(client=client, mapper=mapper, normalizer=normalizer)

# 4. Play PyG: request integers [2, 0] — subset, out of order. Store must reverse
#    them to ids, fetch, and align to THIS order.
attr = TensorAttr(group_name="Account", attr_name="x", index=torch.tensor([2, 0]))
feats = store._get_tensor(attr)

print(f"requested integers [2, 0] -> ids {mapper.to_strings('Account', [2, 0])}")
print(f"returned shape: {tuple(feats.shape)}  (expect (2, 31))")
print(f"dtype: {feats.dtype}")
print(f"row 0 = features for {mapper.to_string('Account', 2)} (first 8 of 31):")
print(f"  {[round(v, 3) for v in feats[0].tolist()[:8]]}")

/Users/abrahamchandy/Documents/Work/companies/2026/TigerGraph/MulePatternLearner/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/abrahamchandy/Documents/Work/companies/2026/TigerGraph/MulePatternLearner/.venv/lib/python3.14/site-packages/torch_geometric/llm/utils/backend_utils.py:26: DeprecationWarning: `torch_geometric.distributed` has been deprecated since 2.7.0 and will no longer be maintained. For distributed training, refer to our tutorials on distributed training at https://pytorch-geometric.readthedocs.io/en/latest/tutorial/distributed.html or cuGraph examples at https://github.com/rapidsai/cugraph-gnn/tree/main/python/cugraph-pyg/cugraph_pyg/examples
  from torch_geometric.distributed import (
/Users/abrahamchandy/Documents/Work/companies/2026/TigerGraph/MulePatternLearner/.venv/lib

4 real account ids: ['A0000066179', 'XM00000053', 'A0000083304', 'XM00001071']

mapper.register -> integers PyG would use: [0, 1, 2, 3]
  int 0 <-> A0000066179

requested integers [2, 0] -> ids ['A0000083304', 'A0000066179']
returned shape: (2, 31)  (expect (2, 31))
dtype: torch.float32
row 0 = features for A0000083304 (first 8 of 31):
  [-0.881, 0.0, -1.37, 0.189, 1.713, -1.527, 0.58, -1.137]


#### Sampler of the graph

In [27]:
import torch
from torch_geometric.sampler import NodeSamplerInput

from mule_pattern_learner.pyg.sampler import TigerGraphHeteroSampler
from mule_pattern_learner.pyg.neighbors import NeighborFanout
from mule_pattern_learner.indexing.node_id_mapper import NodeIDMapper
from mule_pattern_learner.training.seeds_source import fetch_split_seeds

# Real seed ids from the graph.
train_seeds = fetch_split_seeds(client, "train")
seed_ids = tuple(train_seeds.account_ids[:4])   # 4 seeds, small
print(f"4 seed accounts: {seed_ids}\n")

# Build the sampler: it needs the client, the seed_ids (defines index<->id),
# a fresh mapper, and a fanout. allow_val/test=False = strict training regime.
mapper = NodeIDMapper()
fanout = NeighborFanout()
sampler = TigerGraphHeteroSampler(
    client=client,
    seed_ids=seed_ids,
    mapper=mapper,
    fanout=fanout,
    allow_val=False,
    allow_test=False,
)

# PyG calls sample_from_nodes with integer indices into seed_ids. Ask for all 4.
print("running sample_khop_neighborhood on the server (~slow, heavy query)...")
index = NodeSamplerInput(input_id=None, node=torch.tensor([0, 1, 2, 3], dtype=torch.long))
out = sampler.sample_from_nodes(index)
print("done.\n")

# Dissect the HeteroSamplerOutput.
print("HeteroSamplerOutput — the sampled neighborhood structure:")
print(f"  node types returned: {list(out.node.keys())}")
for ntype, tensor in out.node.items():
    print(f"    {ntype:<16}: {tensor.shape[0]} nodes  (integer ids)")
print()
print(f"  edge types returned: {len(out.row)} types")
for etype in out.row:
    print(f"    {etype}: {out.row[etype].shape[0]} edges")
print()

# The mapper now holds this batch's nodes (the sampler registered them).
print(f"mapper after sampling holds {mapper.num_nodes('Account')} Account nodes")
print(f"  (the 4 seeds + their sampled neighbors)")
print(f"  first 4 Account ids (should be the seeds, ordered first):")
print(f"    {mapper.to_strings('Account', [0, 1, 2, 3])}")
print(f"    match seeds: {mapper.to_strings('Account', [0,1,2,3]) == list(seed_ids)}")

4 seed accounts: ('A0000066179', 'XM00000053', 'A0000083304', 'XM00001071')

running sample_khop_neighborhood on the server (~slow, heavy query)...
done.

HeteroSamplerOutput — the sampled neighborhood structure:
  node types returned: ['Account', 'Party', 'Resolved_Entity']
    Account         : 353 nodes  (integer ids)
    Party           : 18 nodes  (integer ids)
    Resolved_Entity : 18 nodes  (integer ids)

  edge types returned: 6 types
    ('Party', 'Party_Account', 'Account'): 27 edges
    ('Resolved_Entity', 'Entity_Party', 'Party'): 18 edges
    ('Party', 'Party_Entity', 'Resolved_Entity'): 18 edges
    ('Account', 'Account_Party', 'Party'): 20 edges
    ('Account', 'Account_Account', 'Account'): 209 edges
    ('Account', 'HAS_PAID', 'Account'): 195 edges

mapper after sampling holds 353 Account nodes
  (the 4 seeds + their sampled neighbors)
  first 4 Account ids (should be the seeds, ordered first):
    ['A0000066179', 'XM00000053', 'A0000083304', 'XM00001071']
    match se

In [42]:
from mule_pattern_learner.pyg.fetch import fetch_has_paid_edges
from mule_pattern_learner.training.seeds_source import fetch_split_seeds

# Real account ids. Use MORE than 4 so we're likely to catch some internal edges
# (HAS_PAID edges only count if BOTH endpoints are in the set).
train_seeds = fetch_split_seeds(client, "train")
account_ids = list(train_seeds.account_ids[:50])
print(f"fetching HAS_PAID edges internal to {len(account_ids)} accounts...\n")

edges = fetch_has_paid_edges(client, account_ids)
print(f"fetch_has_paid_edges returned {len(edges)} edges\n")

if edges:
    # Look at ONE raw edge exactly as it comes back.
    import json
    print("one raw HAS_PAID edge (the shape build_edge_features expects):")
    print(json.dumps(edges[0], indent=2, default=str)[:1500])
    print()

    # Confirm both endpoints are within our account set (the 'internal' contract).
    id_set = set(account_ids)
    sample = edges[:5]
    print("endpoint check (both from_id and to_id should be in our 50 accounts):")
    for e in sample:
        frm, to = e.get("from_id"), e.get("to_id")
        print(f"    {frm} -> {to}   both internal: {frm in id_set and to in id_set}")
else:
    print("no internal HAS_PAID edges among these 50 accounts — try a larger set")
    print("or a set known to transact among themselves.")

# Confirm the empty short-circuit (no server call).
print(f"\nempty-id short-circuit: fetch_has_paid_edges(client, []) -> {fetch_has_paid_edges(client, [])}")

fetching HAS_PAID edges internal to 50 accounts...

fetch_has_paid_edges returned 11 edges

one raw HAS_PAID edge (the shape build_edge_features expects):
{
  "e_type": "HAS_PAID",
  "from_id": "A0000066179",
  "from_type": "Account",
  "to_id": "XM00002114",
  "to_type": "Account",
  "directed": true,
  "attributes": {
    "total_amount": 474.84,
    "total_num_txns": 4,
    "first_txn_date": "2025-03-13 06:09:06",
    "last_txn_date": "2025-04-13 04:41:37",
    "span_days": 30.9392,
    "num_bins": 13,
    "bin_days": 14,
    "amount_bins": [
      0,
      0,
      0,
      0,
      0,
      220.66,
      105.69,
      148.49,
      0,
      0,
      0,
      0,
      0
    ],
    "count_bins": [
      0,
      0,
      0,
      0,
      0,
      2,
      1,
      1,
      0,
      0,
      0,
      0,
      0
    ]
  }
}

endpoint check (both from_id and to_id should be in our 50 accounts):
    A0000066179 -> XM00002114   both internal: True
    A0000066179 -> XM00000249   both int

In [28]:
from mule_pattern_learner.pyg.fetch import fetch_has_paid_edges
from mule_pattern_learner.training.seeds_source import fetch_split_seeds

# Real account ids. Use MORE than 4 so we're likely to catch some internal edges
# (HAS_PAID edges only count if BOTH endpoints are in the set).
train_seeds = fetch_split_seeds(client, "train")
account_ids = list(train_seeds.account_ids[:50])
print(f"fetching HAS_PAID edges internal to {len(account_ids)} accounts...\n")

edges = fetch_has_paid_edges(client, account_ids)
print(f"fetch_has_paid_edges returned {len(edges)} edges\n")

if edges:
    # Look at ONE raw edge exactly as it comes back.
    import json
    print("one raw HAS_PAID edge (the shape build_edge_features expects):")
    print(json.dumps(edges[0], indent=2, default=str)[:1500])
    print()

    # Confirm both endpoints are within our account set (the 'internal' contract).
    id_set = set(account_ids)
    sample = edges[:5]
    print("endpoint check (both from_id and to_id should be in our 50 accounts):")
    for e in sample:
        frm, to = e.get("from_id"), e.get("to_id")
        print(f"    {frm} -> {to}   both internal: {frm in id_set and to in id_set}")
else:
    print("no internal HAS_PAID edges among these 50 accounts — try a larger set")
    print("or a set known to transact among themselves.")

# Confirm the empty short-circuit (no server call).
print(f"\nempty-id short-circuit: fetch_has_paid_edges(client, []) -> {fetch_has_paid_edges(client, [])}")

fetching HAS_PAID edges internal to 50 accounts...

fetch_has_paid_edges returned 11 edges

one raw HAS_PAID edge (the shape build_edge_features expects):
{
  "e_type": "HAS_PAID",
  "from_id": "A0000066179",
  "from_type": "Account",
  "to_id": "XM00002114",
  "to_type": "Account",
  "directed": true,
  "attributes": {
    "total_amount": 474.84,
    "total_num_txns": 4,
    "first_txn_date": "2025-03-13 06:09:06",
    "last_txn_date": "2025-04-13 04:41:37",
    "span_days": 30.9392,
    "num_bins": 13,
    "bin_days": 14,
    "amount_bins": [
      0,
      0,
      0,
      0,
      0,
      220.66,
      105.69,
      148.49,
      0,
      0,
      0,
      0,
      0
    ],
    "count_bins": [
      0,
      0,
      0,
      0,
      0,
      2,
      1,
      1,
      0,
      0,
      0,
      0,
      0
    ]
  }
}

endpoint check (both from_id and to_id should be in our 50 accounts):
    A0000066179 -> XM00002114   both internal: True
    A0000066179 -> XM00000249   both int

In [29]:
# pyg/transform.py : HasPaidEdgeFeatureAttacher
# PyG's remote FeatureStore serves NODE features only. HAS_PAID
# edge features are attached per-batch by a loader transform. 

import torch
from torch_geometric.data import HeteroData
from mule_pattern_learner.pyg.transform import HasPaidEdgeFeatureAttacher, EdgeFeatureError
from mule_pattern_learner.features.edge_spec import flat_edge_dim

MAX_BINS = 13                      # confirmed below by derive_temporal_spec
REF_EPOCH_S = 1_600_000_000.0      # placeholder; real value derived next cell

# The transform shares the backend mapper to turn local Account ints back into
# global ids for the fetch. The empty-edge path is pure-local (no server call),
# so we can verify the zero-edge contract deterministically.
from mule_pattern_learner.indexing.node_id_mapper import NodeIDMapper
mapper = NodeIDMapper()
attacher = HasPaidEdgeFeatureAttacher(
    client=client, mapper=mapper, reference_epoch_s=REF_EPOCH_S, max_bins=MAX_BINS
)

# Build a HeteroData with a HAS_PAID store that has ZERO edges -> transform must
# write a correctly-shaped empty edge_attr (0, flat_edge_dim) and not hit the DB.
empty = HeteroData()
empty["Account"].n_id = torch.tensor([], dtype=torch.long)
empty["Account", "HAS_PAID", "Account"].edge_index = torch.zeros((2, 0), dtype=torch.long)
out = attacher(empty)
attr = out["Account", "HAS_PAID", "Account"].edge_attr
print(f"zero-edge batch -> edge_attr shape {tuple(attr.shape)}  (expect (0, {flat_edge_dim(MAX_BINS)}))")
print(f"  matches flat_edge_dim({MAX_BINS}) = {flat_edge_dim(MAX_BINS)}: {attr.shape[1] == flat_edge_dim(MAX_BINS)}")
print("  (no server call on the empty path — pure shape guarantee)")
print("\nThe non-empty path is exercised end-to-end in the loader cell below,")
print("where real HAS_PAID edges are fetched and aligned row-for-row.")

zero-edge batch -> edge_attr shape (0, 34)  (expect (0, 34))
  matches flat_edge_dim(13) = 34: True
  (no server call on the empty path — pure shape guarantee)

The non-empty path is exercised end-to-end in the loader cell below,
where real HAS_PAID edges are fetched and aligned row-for-row.


In [31]:
# pyg/transform.py + temporal.py : a REAL HAS_PAID edge,
# transformed, and placed on a PyG HeteroData edge store.

import torch
from torch_geometric.data import HeteroData
from torch_geometric.typing import EdgeType

from mule_pattern_learner.pyg.fetch import fetch_has_paid_edges
from mule_pattern_learner.features.temporal import build_edge_features, flat_edge_features
from mule_pattern_learner.features.edge_spec import (
    SCALAR_FEATURE_NAMES, NUM_SCALAR_FEATURES, NUM_BIN_CHANNELS, flat_edge_dim,
)
from mule_pattern_learner.training.seeds_source import fetch_split_seeds

_HAS_PAID: EdgeType = ("Account", "HAS_PAID", "Account")
MAX_BINS = 13                 # re-confirmed by derive_temporal_spec in the next cell
REF_EPOCH_S = 1_600_000_000.0 # placeholder; real value derived next cell

# ── 1. Get REAL internal HAS_PAID edges (both endpoints in the set) ──
train_seeds = fetch_split_seeds(client, "train")
account_ids = list(train_seeds.account_ids[:50])
raw_edges = fetch_has_paid_edges(client, account_ids)
print(f"fetched {len(raw_edges)} real internal HAS_PAID edges among {len(account_ids)} accounts\n")

# ── 2. RAW -> EdgeFeatures (the structured intermediate) ──
ef = build_edge_features(raw_edges, REF_EPOCH_S, MAX_BINS)
print(f"EdgeFeatures: {ef.num_edges} edges")
print(f"  scalar_feats : {tuple(ef.scalar_feats.shape)}   (E, {NUM_SCALAR_FEATURES})")
print(f"  bin_seq      : {tuple(ef.bin_seq.shape)}   (E, {MAX_BINS}, {NUM_BIN_CHANNELS})\n")

# Inspect edge 0 fully: its endpoints, its 8 named scalars, its per-bin grid.
e0_src, e0_dst = ef.src_ids[0], ef.dst_ids[0]
print(f"--- edge 0:  {e0_src} -> {e0_dst} ---")
print("  scalar features (named):")
for name, val in zip(SCALAR_FEATURE_NAMES, ef.scalar_feats[0].tolist()):
    print(f"      {name:<22} {val:>12.4f}")
print(f"\n  bin_seq[0]  (each row = one time-bin: [log1p(amount), count]):")
for i, (amt, cnt) in enumerate(ef.bin_seq[0].tolist()):
    active = "" if (amt != 0.0 or cnt != 0.0) else "   <- empty/padding"
    print(f"      bin {i:>2}: amount={amt:>8.4f}  count={cnt:>5.1f}{active}")

# ── 3. EdgeFeatures -> flat edge_attr (what the model actually consumes) ──
flat = flat_edge_features(ef)   # [E, flat_edge_dim(MAX_BINS)]
k = flat_edge_dim(MAX_BINS)
print(f"\nflat_edge_features -> {tuple(flat.shape)}   (E, flat_edge_dim({MAX_BINS}) = {k})")
print(f"  layout: [ {NUM_SCALAR_FEATURES} scalars | {MAX_BINS} bins x {NUM_BIN_CHANNELS} channels = {MAX_BINS*NUM_BIN_CHANNELS} ]")
print(f"\n  edge 0 full edge_attr row ({k} floats):")
print(f"    scalars [0:{NUM_SCALAR_FEATURES}] : {[round(v,3) for v in flat[0, :NUM_SCALAR_FEATURES].tolist()]}")
print(f"    bins    [{NUM_SCALAR_FEATURES}:{k}]: {[round(v,3) for v in flat[0, NUM_SCALAR_FEATURES:].tolist()]}")

# ── 4. Put it on a REAL PyG object, exactly as the loader's edge store holds it ──
# Map the edge endpoints to local integer indices over the distinct accounts in
# these edges (seeds-agnostic local space, same idea the reindexer uses).
local_ids: list[str] = []
local_of: dict[str, int] = {}
for s, d in zip(ef.src_ids, ef.dst_ids):
    for a in (s, d):
        if a not in local_of:
            local_of[a] = len(local_ids)
            local_ids.append(a)
row = torch.tensor([local_of[s] for s in ef.src_ids], dtype=torch.long)
col = torch.tensor([local_of[d] for d in ef.dst_ids], dtype=torch.long)

data = HeteroData()
data["Account"].n_id = torch.tensor([0] * len(local_ids), dtype=torch.long)  # placeholder ids
data[_HAS_PAID].edge_index = torch.stack([row, col], dim=0)   # [2, E]
data[_HAS_PAID].edge_attr  = flat                              # [E, k]  <- the transformed features

store = data[_HAS_PAID]
print("\n" + "=" * 60)
print("PyG HeteroData edge store — HAS_PAID:")
print(f"  edge_index : {tuple(store.edge_index.shape)}   (2, E)")
print(f"  edge_attr  : {tuple(store.edge_attr.shape)}   (E, {k})")
print(f"  edge_attr.dtype: {store.edge_attr.dtype}")
print(f"  edge_attr width == flat_edge_dim({MAX_BINS}) == model edge_dim: {store.edge_attr.shape[1] == k}")
print(f"  edge 0 in the object:  node {int(row[0])} -> node {int(col[0])}  "
      f"({local_ids[int(row[0])]} -> {local_ids[int(col[0])]})")
print("\nThis edge_attr tensor is byte-identical to what the in-loader attacher")
print("writes; cell N+4 shows the attacher producing it via the shared mapper.")

fetched 11 real internal HAS_PAID edges among 50 accounts

EdgeFeatures: 11 edges
  scalar_feats : (11, 8)   (E, 8)
  bin_seq      : (11, 13, 2)   (E, 13, 2)

--- edge 0:  A0000066179 -> XM00002114 ---
  scalar features (named):
      amount_transferred           6.1651
      transaction_count            1.6094
      active_span_days            30.9392
      active_bin_count             3.0000
      recency_days                 0.0000
      duration_days               30.9392
      active_bin_fraction          0.2308
      amount_per_transaction     118.7100

  bin_seq[0]  (each row = one time-bin: [log1p(amount), count]):
      bin  0: amount=  0.0000  count=  0.0   <- empty/padding
      bin  1: amount=  0.0000  count=  0.0   <- empty/padding
      bin  2: amount=  0.0000  count=  0.0   <- empty/padding
      bin  3: amount=  0.0000  count=  0.0   <- empty/padding
      bin  4: amount=  0.0000  count=  0.0   <- empty/padding
      bin  5: amount=  5.4011  count=  2.0
      bin  6: am

In [30]:
from mule_pattern_learner.tigergraph.derivation import (
    derive_temporal_spec,
    derive_reference_epoch_s,
    GraphTemporalSpec,
)
# NOTE: each of these runs a FULL edge scan on the server — expect a pause.
spec = derive_temporal_spec(client)
ref_epoch_s = derive_reference_epoch_s(client)

print(f"GraphTemporalSpec: max_bins = {spec.max_bins}")
print(f"  derived edge_dim = {spec.edge_dim}  (= flat_edge_dim({spec.max_bins}))")
print(f"  matches the 34 you computed in the edge-spec cell: {spec.edge_dim == 34}")
print()
from datetime import datetime, timezone
print(f"reference_epoch_s = {ref_epoch_s:.0f}")
print(f"  = {datetime.fromtimestamp(ref_epoch_s, tz=timezone.utc)} UTC (latest txn in graph)")
print()
print(f"sanity vs the raw edge you saw earlier (num_bins=13): {spec.max_bins == 13}")
# Pin these for the loader so it's consistent with the data, not a guessed const.
MAX_BINS = spec.max_bins
REF_EPOCH_S = ref_epoch_s

GraphTemporalSpec: max_bins = 13
  derived edge_dim = 34  (= flat_edge_dim(13))
  matches the 34 you computed in the edge-spec cell: True

reference_epoch_s = 1753450140
  = 2025-07-25 13:29:00+00:00 UTC (latest txn in graph)

sanity vs the raw edge you saw earlier (num_bins=13): True


In [33]:
import torch
from mule_pattern_learner.pyg.graph_store import TigerGraphGraphStore
from mule_pattern_learner.indexing.reindex import edge_type_schema
from mule_pattern_learner.indexing.node_id_mapper import NodeIDMapper

# ── (a) metadata: the edge types NodeLoader reads at construction ──
gstore = TigerGraphGraphStore(client=client, mapper=NodeIDMapper())
edge_attrs = gstore.get_all_edge_attrs()
print(f"get_all_edge_attrs -> {len(edge_attrs)} edge types (what NodeLoader reads at build):")
for ea in edge_attrs:
    print(f"    {ea.edge_type}   layout={ea.layout}")
print()
print("PyG triple -> raw GSQL relation name (the single naming bridge):")
for name, triple in edge_type_schema().items():
    print(f"    {str(triple):52s} <- {name!r}")

# ── (b) REAL data: one small page of HAS_PAID edges, store's own parse path ──
RELATION = "HAS_PAID"          # try "Account_Party" or "Party_Entity" to see other types
PAGE_SIZE = 5                  # bounds the export: 5 source nodes + their edges
print("\n" + "=" * 64)
print(f"One bounded page of REAL '{RELATION}' edges (page_size={PAGE_SIZE})")
print("=" * 64)

raw = client.conn.runInstalledQuery(
    "export_edges_by_type",
    {"relation": RELATION, "cursor": "", "page_size": PAGE_SIZE},
)
# Parse with the STORE's own parser, so we're exercising graph_store.py logic,
# not a re-implementation. _parse_page returns (src_ids, dst_ids, next_cursor).
src_ids, dst_ids, next_cursor = gstore._parse_page(raw)
print(f"parsed {len(src_ids)} edges from this page; next_cursor = {next_cursor!r}\n")

print("raw global-id edges (first 10):")
for s, d in list(zip(src_ids, dst_ids))[:10]:
    print(f"    {s}  ->  {d}")

# ── map global ids -> PyG integer indices, exactly as _get_edge_index does ──
mapper = NodeIDMapper()
src_type, dst_type = "Account", "Account"   # HAS_PAID is Account->Account
row = mapper.register(src_type, src_ids)
col = mapper.register(dst_type, dst_ids)
row_t = torch.tensor(row, dtype=torch.long)
col_t = torch.tensor(col, dtype=torch.long)

print(f"\nas a PyG COO edge_index for {RELATION}:")
print(f"  edge_index shape : {tuple(torch.stack([row_t, col_t]).shape)}   (2, {len(row)})")
print(f"  row (src ints)   : {row_t[:10].tolist()}{' ...' if len(row) > 10 else ''}")
print(f"  col (dst ints)   : {col_t[:10].tolist()}{' ...' if len(col) > 10 else ''}")
print(f"  distinct nodes registered: {mapper.num_nodes('Account')}")
print(f"\n  edge 0 round-trip:  int {row_t[0]} -> int {col_t[0]}   "
      f"({mapper.to_string('Account', int(row_t[0]))} -> {mapper.to_string('Account', int(col_t[0]))})")
print("\nThis (row, col) pair is exactly what _get_edge_index returns for the")
print("whole relation — shown here for one page so the relation isn't drained.")

get_all_edge_attrs -> 6 edge types (what NodeLoader reads at build):
    ('Account', 'HAS_PAID', 'Account')   layout=EdgeLayout.COO
    ('Account', 'Account_Account', 'Account')   layout=EdgeLayout.COO
    ('Account', 'Account_Party', 'Party')   layout=EdgeLayout.COO
    ('Party', 'Party_Entity', 'Resolved_Entity')   layout=EdgeLayout.COO
    ('Resolved_Entity', 'Entity_Party', 'Party')   layout=EdgeLayout.COO
    ('Party', 'Party_Account', 'Account')   layout=EdgeLayout.COO

PyG triple -> raw GSQL relation name (the single naming bridge):
    ('Account', 'HAS_PAID', 'Account')                   <- 'HAS_PAID'
    ('Account', 'Account_Account', 'Account')            <- 'Account_Account'
    ('Account', 'Account_Party', 'Party')                <- 'Account_Party'
    ('Party', 'Party_Entity', 'Resolved_Entity')         <- 'Party_Entity'
    ('Resolved_Entity', 'Entity_Party', 'Party')         <- 'Entity_Party'
    ('Party', 'Party_Account', 'Account')                <- 'Party_Account'

On

In [36]:
#  build the loader and pull ONE batch
from typing import cast
from torch import Tensor
from torch_geometric.data import HeteroData
from torch_geometric.typing import EdgeType, NodeType
from mule_pattern_learner.pyg.backend import TigerGraphRemoteBackend
from mule_pattern_learner.training.seeds_source import fetch_split_seeds

_HAS_PAID: EdgeType = ("Account", "HAS_PAID", "Account")

train_seeds = fetch_split_seeds(client, "train")
seed_ids = tuple(train_seeds.account_ids[:8])
print(f"{len(seed_ids)} real train seeds:\n  {list(seed_ids)}")

backend = TigerGraphRemoteBackend(client)
loader = backend.make_loader(
    seed_ids=seed_ids,
    reference_epoch_s=REF_EPOCH_S,
    max_bins=MAX_BINS,
    batch_size=8,
    shuffle=False,
    allow_val=False,
    allow_test=False,
)

print("\npulling one HeteroData batch (server sampling — slow)...")
batch = cast(HeteroData, next(iter(loader)))
print("done.")
print(f"\nbatch object: {type(batch).__name__}")
print(f"node types : {batch.node_types}")
print(f"edge types : {batch.edge_types}")
print(f"batch_size attr (seed count this batch): {getattr(batch, 'batch_size', 'n/a')}")

8 real train seeds:
  ['A0000066179', 'XM00000053', 'A0000083304', 'XM00001071', 'XM00000608', 'A0000095439', 'A0000000231', 'XM00000812']

pulling one HeteroData batch (server sampling — slow)...
done.

batch object: HeteroData
node types : ['Account', 'Party', 'Resolved_Entity']
edge types : [('Account', 'HAS_PAID', 'Account'), ('Account', 'Account_Account', 'Account'), ('Account', 'Account_Party', 'Party'), ('Party', 'Party_Entity', 'Resolved_Entity'), ('Resolved_Entity', 'Entity_Party', 'Party'), ('Party', 'Party_Account', 'Account')]
batch_size attr (seed count this batch): n/a


In [38]:
# node integer-id tensors (from the sampler)
print("node tensors (sampler output, per node type)\n")
for ntype in batch.node_types:
    store = batch[ntype]
    nid = store.n_id
    print(f"  {ntype}")
    print(f"    n_id shape : {tuple(nid.shape)}   ({nid.shape[0]} nodes)")
    print(f"    n_id dtype : {nid.dtype}")
    print(f"    first 8 ids: {nid[:8].tolist()}")
    print()

bsz = int(getattr(batch, "batch_size", len(seed_ids)))
acct_nid = batch["Account"].n_id
print(f"  seed contract: first {bsz} Account rows are the {bsz} seeds.")
print(f"    Account n_id[:{bsz}] = {acct_nid[:bsz].tolist()}")

node tensors (sampler output, per node type)

  Account
    n_id shape : (650,)   (650 nodes)
    n_id dtype : torch.int64
    first 8 ids: [0, 1, 2, 3, 4, 5, 6, 7]

  Party
    n_id shape : (43,)   (43 nodes)
    n_id dtype : torch.int64
    first 8 ids: [0, 1, 2, 3, 4, 5, 6, 7]

  Resolved_Entity
    n_id shape : (43,)   (43 nodes)
    n_id dtype : torch.int64
    first 8 ids: [0, 1, 2, 3, 4, 5, 6, 7]

  seed contract: first 8 Account rows are the 8 seeds.
    Account n_id[:8] = [0, 1, 2, 3, 4, 5, 6, 7]


In [39]:
# edge structure (edge_index) per edge type
# Each edge_index is [2, E]: row 0 src local idx, row 1 dst local idx.
print("edge_index per edge type (graph structure)\n")
total_edges = 0
for etype in batch.edge_types:
    ei = batch[etype].edge_index
    e = int(ei.shape[1])
    total_edges += e
    print(f"  {str(etype):52s}")
    print(f"    edge_index shape : {tuple(ei.shape)}   ({e} edges)")
    print(f"    row[:6] (src)    : {ei[0, :6].tolist()}")
    print(f"    col[:6] (dst)    : {ei[1, :6].tolist()}")
    print()
print(f"  total edges across all relations: {total_edges}")


edge_index per edge type (graph structure)

  ('Account', 'HAS_PAID', 'Account')                  
    edge_index shape : (2, 429)   (429 edges)
    row[:6] (src)    : [509, 5, 169, 207, 4, 2]
    col[:6] (dst)    : [253, 471, 250, 554, 216, 516]

  ('Account', 'Account_Account', 'Account')           
    edge_index shape : (2, 451)   (451 edges)
    row[:6] (src)    : [216, 516, 190, 381, 381, 251]
    col[:6] (dst)    : [352, 582, 236, 105, 506, 201]

  ('Account', 'Account_Party', 'Party')               
    edge_index shape : (2, 45)   (45 edges)
    row[:6] (src)    : [287, 298, 178, 255, 186, 597]
    col[:6] (dst)    : [24, 32, 29, 25, 27, 6]

  ('Party', 'Party_Entity', 'Resolved_Entity')        
    edge_index shape : (2, 43)   (43 edges)
    row[:6] (src)    : [12, 3, 1, 28, 33, 16]
    col[:6] (dst)    : [14, 29, 28, 20, 16, 30]

  ('Resolved_Entity', 'Entity_Party', 'Party')        
    edge_index shape : (2, 43)   (43 edges)
    row[:6] (src)    : [0, 12, 2, 25, 4, 31]
   

('Account','HAS_PAID','Account') (2, 429): 429 payment edges; each column is src→dst, e.g. account 509 paid account 253.

('Account','Account_Account','Account') (2, 451): 451 account-to-account co-occurrence/derived links.

('Account','Account_Party','Party') (2, 45): 45 edges from accounts to their owning parties.

('Party','Party_Entity','Resolved_Entity') (2, 43): 43 edges linking parties to their resolved real-world entity.

('Resolved_Entity','Entity_Party','Party') (2, 43): the same links reversed (entity back to party).

('Party','Party_Account','Account') (2, 61): 61 edges from parties back to their accounts.

In [41]:
# node features x (filled by the FeatureStore)

print("node features x (FeatureStore)\n")
for ntype in batch.node_types:
    store = batch[ntype]
    if hasattr(store, "x"):
        x = store.x
        print(f"  {ntype:16s}: x shape {tuple(x.shape)}  dtype {x.dtype}")
        print(f"      row 0, first 8 feats: {[round(v, 3) for v in x[0, :8].tolist()]}")
    else:
        print(f"  {ntype:16s}: (no x — featureless, learned embedding in model)")




node features x (FeatureStore)

  Account         : x shape (650, 31)  dtype torch.float32
      row 0, first 8 feats: [8.508, 12.97, 5.799, 6.098, 0.008, 5.642, 3.892, 9.85]
  Party           : (no x — featureless, learned embedding in model)
  Resolved_Entity : (no x — featureless, learned embedding in model)


In [43]:
# HAS_PAID edge_attr (filled by the transform)

from mule_pattern_learner.features.edge_spec import flat_edge_dim

hp = batch[_HAS_PAID]
ea = hp.edge_attr
print("HAS_PAID edge_attr (transform output)\n")
print(f"  edge_index shape : {tuple(hp.edge_index.shape)}")
print(f"  edge_attr shape  : {tuple(ea.shape)}")
print(f"  edge_attr dtype  : {ea.dtype}")
print(f"  flat_edge_dim({MAX_BINS}) = {flat_edge_dim(MAX_BINS)}")
print(f"  width matches    : {ea.shape[1] == flat_edge_dim(MAX_BINS)}")
print(f"  rows == #HAS_PAID edges: {ea.shape[0] == hp.edge_index.shape[1]}")
print(f"\n  edge 0 attr (first 12 of {ea.shape[1]} values):")
print(f"    {[round(v, 3) for v in ea[0, :12].tolist()]}")




HAS_PAID edge_attr (transform output)

  edge_index shape : (2, 429)
  edge_attr shape  : (429, 34)
  edge_attr dtype  : torch.float32
  flat_edge_dim(13) = 34
  width matches    : True
  rows == #HAS_PAID edges: True

  edge 0 attr (first 12 of 34 values):
    [4.758, 0.693, 0.0, 1.0, 0.0, 0.0, 0.077, 115.54, 4.758, 1.0, 0.0, 0.0]


In [44]:
# assembled-batch summary (every store, one place)

print("ASSEMBLED HeteroData — all stores\n")
print("NODES:")
for ntype in batch.node_types:
    store = batch[ntype]
    n = int(store.n_id.shape[0])
    xs = tuple(store.x.shape) if hasattr(store, "x") else None
    print(f"  {ntype:16s}: {n:>4} nodes   x={xs}")
print("\nEDGES:")
for etype in batch.edge_types:
    store = batch[etype]
    e = int(store.edge_index.shape[1])
    eas = tuple(store.edge_attr.shape) if hasattr(store, "edge_attr") else None
    print(f"  {str(etype):52s}: {e:>4} edges   edge_attr={eas}")
print("\n  structure (edge_index) + Account x + HAS_PAID edge_attr = model-ready.")
print("  NEXT: repackage into x_dict / edge_index_dict / edge_attr_dict / node_counts,")
print("        then MulePatternModel forward (the model cell).")

ASSEMBLED HeteroData — all stores

NODES:
  Account         :  650 nodes   x=(650, 31)
  Party           :   43 nodes   x=None
  Resolved_Entity :   43 nodes   x=None

EDGES:
  ('Account', 'HAS_PAID', 'Account')                  :  429 edges   edge_attr=(429, 34)
  ('Account', 'Account_Account', 'Account')           :  451 edges   edge_attr=None
  ('Account', 'Account_Party', 'Party')               :   45 edges   edge_attr=None
  ('Party', 'Party_Entity', 'Resolved_Entity')        :   43 edges   edge_attr=None
  ('Resolved_Entity', 'Entity_Party', 'Party')        :   43 edges   edge_attr=None
  ('Party', 'Party_Account', 'Account')               :   61 edges   edge_attr=None

  structure (edge_index) + Account x + HAS_PAID edge_attr = model-ready.
  NEXT: repackage into x_dict / edge_index_dict / edge_attr_dict / node_counts,
        then MulePatternModel forward (the model cell).


In [45]:
# ONE forward pass (untrained) -> logits

import torch
from mule_pattern_learner.pyg.model import MulePatternModel
from mule_pattern_learner.features.nodes import NUM_ACCOUNT_FEATURES
from mule_pattern_learner.features.edge_spec import flat_edge_dim

# Repackage the batch into the dicts model.forward expects.
x_dict: dict[NodeType, Tensor] = {"Account": batch["Account"].x}
node_counts: dict[NodeType, int] = {
    ntype: int(batch[ntype].n_id.shape[0]) for ntype in batch.node_types
}
edge_index_dict: dict[EdgeType, Tensor] = {
    etype: batch[etype].edge_index for etype in batch.edge_types
}
edge_attr_dict: dict[EdgeType, Tensor] = {_HAS_PAID: batch[_HAS_PAID].edge_attr}

model = MulePatternModel(
    account_in_dim=NUM_ACCOUNT_FEATURES,   # 31
    edge_dim=flat_edge_dim(MAX_BINS),      # 34
    hidden_dim=64,
    heads=4,
)
print(f"  Account x in       : {tuple(x_dict['Account'].shape)}  [N, {NUM_ACCOUNT_FEATURES}]")
print(f"  HAS_PAID edge_attr : {tuple(edge_attr_dict[_HAS_PAID].shape)}  [E, {flat_edge_dim(MAX_BINS)}]")
print(f"  node counts        : {node_counts}")

_ = model.eval()
with torch.no_grad():
    logits = cast(Tensor, model(x_dict, edge_index_dict, edge_attr_dict, node_counts))

n_account = node_counts.get("Account", 0)
print()
print(f"  logits shape       : {tuple(logits.shape)}  (expect [{n_account}])")
print(f"  one logit/account  : {int(logits.shape[0]) == n_account}")
print(f"  first logits       : {[round(v, 4) for v in logits[:5].tolist()]}")
print()
print("One mule-likelihood logit per Account. NEXT (training boundary):")


  Account x in       : (650, 31)  [N, 31]
  HAS_PAID edge_attr : (429, 34)  [E, 34]
  node counts        : {'Account': 650, 'Party': 43, 'Resolved_Entity': 43}

  logits shape       : (650,)  (expect [650])
  one logit/account  : True
  first logits       : [-21.7411, -16.8716, -7.0739, -12.2499, -20.7205]

One mule-likelihood logit per Account. NEXT (training boundary):
  training/loss.py  -> NonNegativePULoss(prior=...)
  training/loop.py  -> AdamW + early stopping on val PAUC


/Users/abrahamchandy/Documents/Work/companies/2026/TigerGraph/MulePatternLearner/.venv/lib/python3.14/site-packages/torch_geometric/inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore


In [46]:
import torch
from mule_pattern_learner.training.loss import NonNegativePULoss

PRIOR = 0.003          # true mule fraction (pi). Extreme imbalance, as in this data.
loss_fn = NonNegativePULoss(prior=PRIOR, positive_weight=0.5)  # see docstring: raise > prior

# Fake batch of 10 account logits + pu_labels (1 = revealed positive, 0 = unlabeled).
logits = torch.tensor([ 2.5, -1.0,  0.3,  3.1, -2.0, -0.5,  1.2, -3.0,  0.8, -1.5])
targets = torch.tensor([   1,    0,    0,    1,    0,    0,    0,    0,    0,    0])
print(f"prior (pi)      : {PRIOR}")
print(f"positive_weight : 0.5")
print(f"logits  : {logits.tolist()}")
print(f"targets : {targets.tolist()}   ({int(targets.sum())} positives, {int((targets==0).sum())} unlabeled)\n")

train_loss, objective = loss_fn(logits, targets)
print(f"train_loss (backprop target): {train_loss.item():.6f}")
print(f"objective  (always-unclamped, for monitoring): {objective.item():.6f}")
print(f"clamp fired (train_loss != objective): {abs(train_loss.item() - objective.item()) > 1e-9}")

# Force the clamp: unlabeled points scored very LOW + positives HIGH drives the
# estimated negative_risk below 0 (overfitting signal). nnPU then replaces the
# backprop target with gamma*(-negative_risk) to push it back up. Easiest to SEE
# with a larger prior, so use prior=0.3 just for this illustration.
print("\n-- forcing the nnPU correction (negative_risk < 0) --")
clamp_fn = NonNegativePULoss(prior=0.3, positive_weight=0.5)
clamp_logits = torch.tensor([6.0, -6.0, -6.0, 6.0, -6.0, -6.0, -6.0, -6.0, -6.0, -6.0])
tl2, obj2 = clamp_fn(clamp_logits, targets)
print(f"train_loss: {tl2.item():.6f}   objective: {obj2.item():.6f}")
print(f"clamp fired: {abs(tl2.item() - obj2.item()) > 1e-9}  "
      f"(objective went negative -> backprop switches to push negative_risk up)")

prior (pi)      : 0.003
positive_weight : 0.5
logits  : [2.5, -1.0, 0.30000001192092896, 3.0999999046325684, -2.0, -0.5, 1.2000000476837158, -3.0, 0.800000011920929, -1.5]
targets : [1, 0, 0, 1, 0, 0, 0, 0, 0, 0]   (2 positives, 8 unlabeled)

train_loss (backprop target): 0.405480
objective  (always-unclamped, for monitoring): 0.405480
clamp fired (train_loss != objective): False

-- forcing the nnPU correction (negative_risk < 0) --
train_loss: 0.296786   objective: -0.295549
clamp fired: True  (objective went negative -> backprop switches to push negative_risk up)


In [47]:
# ONE optimizer step (the training boundary).

import torch
from torch.optim import AdamW
from torch_geometric.typing import EdgeType, NodeType
from torch import Tensor
from typing import cast
from mule_pattern_learner.pyg.model import MulePatternModel
from mule_pattern_learner.features.nodes import NUM_ACCOUNT_FEATURES
from mule_pattern_learner.features.edge_spec import flat_edge_dim

_HAS_PAID: EdgeType = ("Account", "HAS_PAID", "Account")

# Repackage the batch (same as the model cell).
x_dict = {"Account": batch["Account"].x}
node_counts = {nt: int(batch[nt].n_id.shape[0]) for nt in batch.node_types}
edge_index_dict = {et: batch[et].edge_index for et in batch.edge_types}
edge_attr_dict = {_HAS_PAID: batch[_HAS_PAID].edge_attr}

# pu_label for the SEED accounts (the model scores all accounts, but loss is on
# seeds — the first `batch_size` Account rows). Pull labels from the seed set.
bsz = int(getattr(batch, "batch_size", len(seed_ids)))
seed_pu = torch.tensor(
    [train_seeds.pu_label_of[seed_ids[i]] for i in range(bsz)], dtype=torch.long
)
print(f"seed pu_labels (first {bsz} accounts): {seed_pu.tolist()}  "
      f"({int(seed_pu.sum())} positive)")

model = MulePatternModel(
    account_in_dim=NUM_ACCOUNT_FEATURES, edge_dim=flat_edge_dim(MAX_BINS),
    hidden_dim=64, heads=4,
)
opt = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = NonNegativePULoss(prior=0.003, positive_weight=0.5)

# Snapshot one weight to prove the step changed it.
w_before = model._head.weight.detach().clone()

_ = model.train()
opt.zero_grad()
logits = cast(Tensor, model(x_dict, edge_index_dict, edge_attr_dict, node_counts))
seed_logits = logits[:bsz]                       # loss on seeds only
train_loss, objective = loss_fn(seed_logits, seed_pu)
train_loss.backward()
opt.step()

w_after = model._head.weight.detach()
print(f"\nforward -> logits shape {tuple(logits.shape)}; seed logits {tuple(seed_logits.shape)}")
print(f"train_loss: {train_loss.item():.6f}   objective: {objective.item():.6f}")
print(f"readout weight changed after step: {not torch.equal(w_before, w_after)}")
print(f"  max |delta| in head weight: {(w_after - w_before).abs().max().item():.3e}")
print("\nThat is one training step. loop.py repeats this over batches/epochs,")
print("evaluating ValScores.roc_auc (Proxy AUC) on the val split for early stopping.")

seed pu_labels (first 8 accounts): [0, 0, 0, 0, 0, 0, 0, 0]  (0 positive)

forward -> logits shape (650,); seed logits (8,)
train_loss: 0.675318   objective: 0.675318
readout weight changed after step: True
  max |delta| in head weight: 1.000e-03

That is one training step. loop.py repeats this over batches/epochs,
evaluating ValScores.roc_auc (Proxy AUC) on the val split for early stopping.


In [48]:
import inspect

# 1. edge_spec: derivation.py needs flat_edge_dim(max_bins) -> int (the edge
#    feature width). Let's see its signature and what it returns for a sample.
from mule_pattern_learner.features.edge_spec import flat_edge_dim
print("flat_edge_dim signature:", inspect.signature(flat_edge_dim))
for sample_bins in (1, 13, 100):
    print(f"  flat_edge_dim({sample_bins}) = {flat_edge_dim(sample_bins)}")
print()

# 2. temporal: transform.py needs build_edge_features + flat_edge_features.
#    Confirm they import and show their signatures (don't call yet — needs data).
from mule_pattern_learner.features.temporal import (
    build_edge_features,
    flat_edge_features,
)
print("build_edge_features signature:", inspect.signature(build_edge_features))
print("flat_edge_features signature :", inspect.signature(flat_edge_features))
print()

# 3. nodes: feature_store.py + train.py need these. Confirm + show the feature count.
from mule_pattern_learner.features.nodes import (
    NUM_ACCOUNT_FEATURES,
    FeatureNormalizer,
    build_account_features,
    normalizer_from_features,
)
print("NUM_ACCOUNT_FEATURES :", NUM_ACCOUNT_FEATURES)
print("build_account_features signature:", inspect.signature(build_account_features))
print("FeatureNormalizer fields         :", FeatureNormalizer.__annotations__ if hasattr(FeatureNormalizer, "__annotations__") else "(inspect below)")

print("\nAll features-layer imports resolved.")

flat_edge_dim signature: (max_bins: int) -> int
  flat_edge_dim(1) = 10
  flat_edge_dim(13) = 34
  flat_edge_dim(100) = 208

build_edge_features signature: (edges: collections.abc.Sequence[object], reference_epoch_s: float, max_bins: int) -> mule_pattern_learner.features.temporal.EdgeFeatures
flat_edge_features signature : (features: mule_pattern_learner.features.temporal.EdgeFeatures) -> torch.Tensor

NUM_ACCOUNT_FEATURES : 31
build_account_features signature: (vertices: collections.abc.Sequence[object]) -> mule_pattern_learner.features.nodes.NodeFeatures
FeatureNormalizer fields         : {'mean': <class 'torch.Tensor'>, 'std': <class 'torch.Tensor'>}

All features-layer imports resolved.


## 7 · Masking — SYNTHETIC ONLY

Applies the PU mask (`pu_label` / `true_label` / `bucket`) and the party-grouped,
**positive-stratified** split, then writes `is_train/is_val/is_test/pu_label` back
to Accounts and the answer-key parquet. **This step does not exist in production**
(real labels replace it). Run via the script so its CLI args apply.

> With the stratified split, val is guaranteed revealed positives. Use
> `--reveal-prevalence 0.10` for a healthier count.

In [ ]:
# Masking is synthetic scaffolding with a CLI; invoke the script rather than
# importing its main(). This MUTATES split flags + writes the parquet.
RUN_MASKING = False  # flip to True to (re)mask
if RUN_MASKING:
    import subprocess
    cmd = [sys.executable, "scripts/pipeline/after_load/masking.py",
           "--reveal-prevalence", "0.10"]
    print("running:", " ".join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout[-2000:])
    if res.returncode != 0:
        print("STDERR:", res.stderr[-1000:])
else:
    print("Skipped (RUN_MASKING=False). Split flags + parquet already present.")

## 8 · Inspect the split seeds (the fail-fast check, interactively)

Before training, confirm each split has revealed positives — the exact guard
`train.py` enforces. With the stratified split, **val pos should be ≥ 1**.

In [ ]:
from mule_pattern_learner.training.seeds_source import fetch_split_seeds

for split in ("train", "val", "test"):
    s = fetch_split_seeds(client, split)
    print(f"{split:5s}: {len(s.account_ids):>7} accounts | "
          f"revealed positives = {s.num_positives}")
# If val positives == 0, re-mask (cell 7) with stratify + higher prevalence.

## 9 · Training a few real batches

A miniature of `mule-train`: build loaders, run a couple of epochs to confirm the
wiring (forward → nnPU loss → backward → PAUC validation → best-checkpoint save).
For a full run, use the `mule-train --estimated-mules N` console script instead.

In [ ]:
# Derive pi the same way the entrypoint does (estimated total mules / accounts).
ESTIMATED_MULES = 216  # synthetic "we guessed right"; production = your estimate
total_accounts = client.conn.getVertexCount("Account", realtime=True)
assert isinstance(total_accounts, int)
prior = ESTIMATED_MULES / total_accounts
print(f"pi = {prior:.6f}  ({ESTIMATED_MULES} / {total_accounts})")

In [ ]:
# This block mirrors train.main() at small scale. It is intentionally a SMOKE:
# 2 epochs, so you can watch one train+val cycle. Full training => mule-train.
RUN_TRAIN_SMOKE = False  # flip on to actually train a couple epochs
if RUN_TRAIN_SMOKE:
    import math
    from collections.abc import Iterable, Iterator
    from typing import cast
    from torch_geometric.data import HeteroData
    from mule_pattern_learner.pyg.backend import TigerGraphRemoteBackend
    from mule_pattern_learner.pyg.model import MulePatternModel
    from mule_pattern_learner.pyg.neighbors import NeighborFanout
    from mule_pattern_learner.device import select_device
    from mule_pattern_learner.training.loop import TrainConfig, fit
    from mule_pattern_learner.training.seeds import SeedPool, epoch_batches

    backend = TigerGraphRemoteBackend(client)
    mapper = backend.mapper
    device = select_device()
    train_seeds = fetch_split_seeds(client, "train")
    val_seeds = fetch_split_seeds(client, "val")
    pu_label_of = dict(train_seeds.pu_label_of); pu_label_of.update(val_seeds.pu_label_of)
    pool = SeedPool(
        positives=tuple(a for a, y in train_seeds.pu_label_of.items() if y == 1),
        unlabeled=tuple(a for a, y in train_seeds.pu_label_of.items() if y == 0),
    )
    fanout = NeighborFanout()
    def to_ids(ints): return mapper.to_strings("Account", ints)
    def train_loader():
        for b in epoch_batches(pool, batch_size=512, positives_per_batch=32, seed=1337):
            yield from cast(Iterable[HeteroData], backend.make_loader(
                seed_ids=b, reference_epoch_s=reference_epoch_s, max_bins=spec.max_bins,
                fanout=fanout, batch_size=len(b), shuffle=False,
                allow_val=False, allow_test=False))
    def val_loader():
        yield from cast(Iterable[HeteroData], backend.make_loader(
            seed_ids=val_seeds.account_ids, reference_epoch_s=reference_epoch_s,
            max_bins=spec.max_bins, fanout=fanout, batch_size=512, shuffle=False,
            allow_val=True, allow_test=False))
    model = MulePatternModel(account_in_dim=31, edge_dim=spec.edge_dim)
    cfg = TrainConfig(prior=prior, max_epochs=2, patience=99, eval_k=50)
    result = fit(model=model, make_train_loader=train_loader, make_val_loader=val_loader,
                 pu_label_of=pu_label_of, mapper_to_ids=to_ids, config=cfg, device=device)
    print("best epoch:", result.best_epoch, "best val PAUC:", round(result.best_val_pauc, 4))
else:
    print("Skipped (RUN_TRAIN_SMOKE=False). Use `mule-train --estimated-mules 216` for a full run.")

## 10 · Hidden-mule evaluation — SYNTHETIC ONLY

After a real `mule-train` run produces a checkpoint, score generalization to
HIDDEN mules against the answer-key parquet. Run the script (auto-discovers the
latest checkpoint + parquet).

In [ ]:
RUN_EVAL = False  # flip on AFTER you have a checkpoint in models/
if RUN_EVAL:
    import subprocess
    res = subprocess.run([sys.executable, "scripts/experiments/evaluate_hidden.py"],
                         capture_output=True, text=True)
    print(res.stdout[-2000:])
    if res.returncode != 0:
        print("STDERR:", res.stderr[-1000:])
else:
    print("Skipped (RUN_EVAL=False). Train first, then flip on.")

---
### Refactor notes (things to look at while exploring)

- **`gsql_install`** now centralizes installs. The 7 old `_gsql` helpers in
  `scripts/experiments/` & `scripts/demos/` can be deleted and re-pointed here.
- **Registry vs installed names**: cell 1 shows the 10 mismatches. A future
  cleanup could store the installed name in `gsql_paths` directly.
- **The destructive gates** (`RUN_DESTRUCTIVE`, `RUN_FEATURES`, `RUN_MASKING`,
  `RUN_TRAIN_SMOKE`, `RUN_EVAL`) are the seams a real orchestrator would turn
  into ordered, idempotent pipeline steps (Makefile target or Python runner).